# Parameter Golf — Lappie AI PyTorch Validation
Smoke test to verify our train_gpt.py runs on CUDA before deploying to H100s.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Step 1: Clone our fork and install deps
!git clone -b lappie-submission https://github.com/lappiecto/parameter-golf.git
%cd parameter-golf
!pip install -q sentencepiece zstandard huggingface-hub datasets

In [ ]:
# Step 2: Download minimal data (1 shard for smoke test)
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

In [ ]:
# Step 3: Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Step 4: Quick syntax check of our submission script
import ast
with open('records/track_10min_16mb/lappie-submission/train_gpt.py') as f:
    code = f.read()
ast.parse(code)
print(f"Syntax OK — {len(code.splitlines())} lines")

In [ ]:
# Step 5: Copy submission script to working directory
!cp records/track_10min_16mb/lappie-submission/train_gpt.py train_gpt_submission.py

In [ ]:
# Step 6: Run smoke test — 50 iterations on 1 GPU
# T4 has 16GB VRAM so we use smaller batch to fit
!RUN_ID=colab_smoke \
  ITERATIONS=50 \
  TRAIN_BATCH_TOKENS=8192 \
  VAL_BATCH_SIZE=8192 \
  TRAIN_SEQ_LEN=1024 \
  TRAIN_LOG_EVERY=10 \
  VAL_LOSS_EVERY=0 \
  MAX_WALLCLOCK_SECONDS=300 \
  SWA_ENABLED=0 \
  QAT_ENABLED=0 \
  WARMDOWN_ITERS=200 \
  torchrun --standalone --nproc_per_node=1 train_gpt_submission.py

In [ ]:
# Step 7: Check results
!grep -E "model_params|val_bpb|final_int8|serialized|error|Error|Traceback" logs/colab_smoke.txt

## What to look for

**SUCCESS** means:
- `model_params:28255422` — correct param count
- `step:50/50 train_loss:X.XXXX` — training completed
- `final_int8_zlib_roundtrip val_bpb:X.XXXX` — quantisation + eval worked
- `serialized_model_int8_zlib:XXXXX bytes` — artifact produced and under 16MB

**FAILURE** means:
- Any `Traceback` or `Error` — note the exact error for debugging
- `CUDA out of memory` — reduce TRAIN_BATCH_TOKENS or TRAIN_SEQ_LEN

If this passes, the script is ready for 8xH100 on RunPod.

In [ ]:
# Step 8: Check artifact size
import os
for f in os.listdir('logs'):
    if f.endswith('.ptz'):
        size = os.path.getsize(f'logs/{f}')
        print(f"{f}: {size:,} bytes ({size/1_000_000:.2f} MB) — {'PASS' if size < 16_000_000 else 'FAIL: OVER 16MB'}")